<a href="https://colab.research.google.com/github/whocoppin-15/agri-trading/blob/main/projet_middle_office.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Génération de données factices sous forme de tableaux**

> Ici on crée un tableau avec des données pour **simuler** un tableau de trades, qui pourraient être importés depuis un fichier `.csv` ou `.xls`

In [186]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Configuration pour l'affichage de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# --- Sample Data Definitions (Added to fix NameError) ---

# 1. df_physical_positions
start_date = datetime(2026, 1, 1)
num_days = 30
dates = [start_date + timedelta(days=i) for i in range(num_days)]

products = ['Sucre', 'Éthanol', 'Blé']
position_types = ['Long', 'Short']

physical_positions_data = []
for date in dates:
    for product_name in products:
        for _ in range(np.random.randint(1, 3)): # 1 or 2 positions per day/product
            position_type = np.random.choice(position_types)
            quantity = np.random.randint(100, 1000) * (1 if position_type == 'Long' else -1)
            price = np.random.uniform(200, 500)
            currency = np.random.choice(['USD', 'EUR'])
            physical_positions_data.append({
                'Date': date,
                'Produit': product_name,
                'Type_Position': 'Physique',
                'Sens': position_type,
                'Quantité': quantity,
                'Prix_Unitaire': price,
                'Devise_Prix': currency
            })
df_physical_positions = pd.DataFrame(physical_positions_data)

# 2. df_financial_hedges
financial_hedges_data = []
future_names = ['Sucre_Future_Dec26', 'Éthanol_Future_Jan27', 'Blé_Future_Mar26']
for date in dates:
    for future_name in future_names:
        for _ in range(np.random.randint(0, 2)): # 0 or 1 hedge per day/future
            position_type = np.random.choice(position_types)
            quantity = np.random.randint(50, 500) * (1 if position_type == 'Long' else -1)
            price = np.random.uniform(250, 550)
            currency = np.random.choice(['USD', 'EUR'])
            financial_hedges_data.append({
                'Date': date,
                'Instrument': future_name,
                'Type_Position': 'Financier',
                'Sens': position_type,
                'Quantité': quantity,
                'Prix_Unitaire': price,
                'Devise_Prix': currency
            })
df_financial_hedges = pd.DataFrame(financial_hedges_data)

# 3. df_market_data
market_data = []

# Spot prices for physical products
for date in dates:
    for product_name in products:
        market_data.append({
            'Date': date,
            'Instrument': product_name,
            'Prix': np.random.uniform(210, 510) # Spot price
        })

# Future prices for financial instruments
for date in dates:
    for future_name in future_names:
        market_data.append({
            'Date': date,
            'Instrument': future_name,
            'Prix': np.random.uniform(260, 560) # Future price
        })

# FX rates
for date in dates:
    market_data.append({
        'Date': date,
        'Instrument': 'USD/EUR',
        'Prix': np.random.uniform(0.90, 0.95) # USD to EUR rate
    })
df_market_data = pd.DataFrame(market_data)

display(df_physical_positions.head())
display(df_financial_hedges.head())
display(df_market_data.head())

,Date,Produit,Type_Position,Sens,Quantité,Prix_Unitaire,Devise_Prix
0,2026-01-01,Sucre,Physique,Long,106,411.65,USD
1,2026-01-01,Sucre,Physique,Long,841,303.60,EUR
2,2026-01-01,Éthanol,Physique,Long,434,291.31,EUR
3,2026-01-01,Éthanol,Physique,Short,-644,260.51,USD
4,2026-01-01,Blé,Physique,Long,878,413.00,USD


,Date,Instrument,Type_Position,Sens,Quantité,Prix_Unitaire,Devise_Prix
0,2026-01-01,Sucre_Future_Dec26,Financier,Short,-303,396.49,USD
1,2026-01-01,Éthanol_Future_Jan27,Financier,Long,243,401.66,EUR
2,2026-01-02,Éthanol_Future_Jan27,Financier,Short,-222,515.12,EUR
3,2026-01-03,Éthanol_Future_Jan27,Financier,Short,-149,332.80,EUR
4,2026-01-04,Sucre_Future_Dec26,Financier,Long,176,285.21,EUR


,Date,Instrument,Prix
0,2026-01-01,Sucre,461.85
1,2026-01-01,Éthanol,243.36
2,2026-01-01,Blé,259.90
3,2026-01-02,Sucre,435.47
4,2026-01-02,Éthanol,383.01


In [187]:
pd.options.display.float_format = '{:,.2f}'.format

### **1. Calcul de la position nette et du PnL MTM**

Pour calculer la position nette et le PnL MTM :

- On doit d'abord **fusionner** les données de positions avec les prix de marché correspondants.

- Nous allons ensuite **consolider** les quantités et **évaluer** le PnL pour chaque journée.

In [188]:
df_physical_positions_merged = df_physical_positions.rename(columns={'Produit': 'Instrument'})
df_all_positions = pd.concat([df_physical_positions_merged, df_financial_hedges], ignore_index=True)
df_all_positions['Date'] = pd.to_datetime(df_all_positions['Date'])
df_market_data['Date'] = pd.to_datetime(df_market_data['Date'])

df_positions_with_prices = pd.merge(
    df_all_positions,
    df_market_data,
    on=['Date', 'Instrument'],
    how='left'
)

df_fx_rates = df_market_data[df_market_data['Instrument'] == 'USD/EUR'][['Date', 'Prix']]
df_fx_rates = df_fx_rates.rename(columns={'Prix': 'Taux_USD_EUR'})

df_positions_with_prices = pd.merge(
    df_positions_with_prices,
    df_fx_rates,
    on='Date',
    how='left'
)

# Calcul de la valeur de chaque position dans sa devise d'origine
df_positions_with_prices['Valeur_Position_Origine'] = df_positions_with_prices['Quantité'] * df_positions_with_prices['Prix_Unitaire']

# conversion en EUR
def convert_to_eur(row):
    if row['Devise_Prix'] == 'USD':
        if pd.isna(row['Taux_USD_EUR']):
            print(f"Warning: Missing Taux_USD_EUR for date {row['Date']}. Using 1.0 as default for now.") # flag
            return row['Valeur_Position_Origine'] * 1.0 # flag
        return row['Valeur_Position_Origine'] * row['Taux_USD_EUR']
    elif row['Devise_Prix'] == 'EUR':
        return row['Valeur_Position_Origine']
    return np.nan # erreur si cela apparait

df_positions_with_prices['Valeur_EUR'] = df_positions_with_prices.apply(convert_to_eur, axis=1)

display(df_positions_with_prices)

,Date,Instrument,Type_Position,Sens,Quantité,Prix_Unitaire,Devise_Prix,Prix,Taux_USD_EUR,Valeur_Position_Origine,Valeur_EUR
0,2026-01-01,Sucre,Physique,Long,106,411.65,USD,461.85,0.91,"43,635.27","39,496.00"
1,2026-01-01,Sucre,Physique,Long,841,303.60,EUR,461.85,0.91,"255,323.55","255,323.55"
2,2026-01-01,Éthanol,Physique,Long,434,291.31,EUR,243.36,0.91,"126,427.90","126,427.90"
3,2026-01-01,Éthanol,Physique,Short,-644,260.51,USD,243.36,0.91,"-167,768.60","-151,853.98"
4,2026-01-01,Blé,Physique,Long,878,413.00,USD,259.90,0.91,"362,612.29","328,214.68"
...,...,...,...,...,...,...,...,...,...,...,...
170,2026-01-25,Sucre_Future_Dec26,Financier,Short,-200,365.96,EUR,358.73,0.90,"-73,191.29","-73,191.29"
171,2026-01-26,Sucre_Future_Dec26,Financier,Short,-498,343.88,USD,448.20,0.91,"-171,254.06","-155,882.37"
172,2026-01-26,Éthanol_Future_Jan27,Financier,Short,-425,285.54,USD,532.93,0.91,"-121,355.48","-110,462.66"
173,2026-01-27,Éthanol_Future_Jan27,Financier,Long,132,452.97,EUR,373.87,0.90,"59,791.86","59,791.86"


In [189]:
df_one_position = df_positions_with_prices.groupby(['Date', 'Instrument', 'Devise_Prix']).agg({
    'Quantité': 'sum',
    'Valeur_EUR': 'sum' # Valeur MTM de la position nette
}).reset_index()

df_one_position = df_one_position.rename(columns={'Quantité': 'One_Position_Nette_Quantité', 'Valeur_EUR': 'MTM_EUR'}) # Ajout de cette ligne pour renommer la colonne
df_one_position = df_one_position[df_one_position['Instrument'] != 'USD/EUR']

df_one_position = df_one_position.sort_values(by=['Instrument', 'Date'])
df_one_position['MTM_EUR_Prev'] = df_one_position.groupby('Instrument')['MTM_EUR'].shift(1)
df_one_position['PnL_MTM_Journalier_EUR'] = df_one_position['MTM_EUR'] - df_one_position['MTM_EUR_Prev']

display(df_one_position)

,Date,Instrument,Devise_Prix,One_Position_Nette_Quantité,MTM_EUR,MTM_EUR_Prev,PnL_MTM_Journalier_EUR
0,2026-01-01,Blé,USD,878,"328,214.68",NaN,NaN
7,2026-01-02,Blé,USD,-329,"-126,152.93","328,214.68","-454,367.61"
11,2026-01-03,Blé,EUR,-258,"-99,381.04","-126,152.93","26,771.89"
15,2026-01-04,Blé,EUR,-182,"-45,282.65","-99,381.04","54,098.39"
16,2026-01-04,Blé,USD,349,"84,078.97","-45,282.65","129,361.62"
...,...,...,...,...,...,...,...
96,2026-01-19,Éthanol_Future_Jan27,USD,120,"51,607.14","-100,136.94","151,744.07"
101,2026-01-20,Éthanol_Future_Jan27,EUR,-225,"-78,354.59","51,607.14","-129,961.73"
112,2026-01-22,Éthanol_Future_Jan27,EUR,340,"169,441.40","-78,354.59","247,795.99"
132,2026-01-26,Éthanol_Future_Jan27,USD,-425,"-110,462.66","169,441.40","-279,904.07"


In [190]:
df_usd_positions = df_positions_with_prices[df_positions_with_prices['Devise_Prix'] == 'USD'].copy()

df_usd_exposure = df_usd_positions.groupby('Date')['Valeur_Position_Origine'].sum().reset_index()
df_usd_exposure = df_usd_exposure.rename(columns={'Valeur_Position_Origine': 'Exposition_USD'})

df_usd_exposure = pd.merge(
    df_usd_exposure,
    df_fx_rates,
    on='Date',
    how='left'
)

display(df_usd_exposure)

,Date,Exposition_USD,Taux_USD_EUR
0,2026-01-01,"118,342.75",0.91
1,2026-01-02,"-136,402.70",0.92
2,2026-01-04,"209,987.26",0.93
3,2026-01-05,"518,643.47",0.95
4,2026-01-06,"-213,721.51",0.90
5,2026-01-07,"351,133.00",0.95
6,2026-01-08,"-364,174.32",0.94
7,2026-01-09,"-806,691.78",0.91
8,2026-01-10,"713,939.87",0.91
9,2026-01-11,"83,533.94",0.91


### **2. Contrôle du Middle Office & Décomposition du P&L (P&L Explain)**

Ce module vise à **expliquer** la variation du P&L entre J et J-1 (effet Prix, effet FX).

- Effet prix : changement du prix de marché sur les positions existantes.
- Effet FX : impact de la variation du taux de change sur les positions en USD (*non présenté ici*)

In [191]:
df_daily_pnl = df_one_position.groupby('Date').agg({
    'MTM_EUR': 'sum',
    'PnL_MTM_Journalier_EUR': 'sum' # Somme des PnL MTM journaliers par instrument
}).reset_index()

df_daily_pnl = df_daily_pnl.rename(columns={'MTM_EUR': 'MTM_Total_EUR', 'PnL_MTM_Journalier_EUR': 'PnL_Total_MTM_EUR'})

df_fx_rates['Taux_USD_EUR_Prev'] = df_fx_rates['Taux_USD_EUR'].shift(1)

df_daily_pnl = pd.merge(df_daily_pnl, df_usd_exposure[['Date', 'Exposition_USD']], on='Date', how='left')
df_daily_pnl = pd.merge(df_daily_pnl, df_fx_rates[['Date', 'Taux_USD_EUR', 'Taux_USD_EUR_Prev']], on='Date', how='left')

df_daily_pnl['Exposition_USD_Prev'] = df_daily_pnl['Exposition_USD'].shift(1) # Exposition du jour précédent

df_daily_pnl['PnL_FX_Carry_EUR'] = df_daily_pnl['Exposition_USD_Prev'] * (df_daily_pnl['Taux_USD_EUR'] - df_daily_pnl['Taux_USD_EUR_Prev'])

df_daily_pnl['PnL_Effet_Prix_EUR'] = df_daily_pnl['PnL_Total_MTM_EUR'] - df_daily_pnl['PnL_FX_Carry_EUR'].fillna(0)

display(df_daily_pnl[['Date', 'MTM_Total_EUR', 'PnL_Total_MTM_EUR', 'PnL_Effet_Prix_EUR', 'PnL_FX_Carry_EUR']])

,Date,MTM_Total_EUR,PnL_Total_MTM_EUR,PnL_Effet_Prix_EUR,PnL_FX_Carry_EUR
0,2026-01-01,"586,470.80","-494,109.43","-494,109.43",NaN
1,2026-01-02,"286,049.92","-27,409.44","-29,742.82","2,333.38"
2,2026-01-03,"34,286.05","-251,763.87","-253,546.92","1,783.05"
3,2026-01-04,"-107,834.52","270,378.49","270,378.49",NaN
4,2026-01-05,"533,401.29","-14,772.06","-18,618.42","3,846.37"
5,2026-01-06,"120,252.75","-110,487.55","-85,404.12","-25,083.43"
6,2026-01-07,"645,341.68","348,057.68","358,447.11","-10,389.43"
7,2026-01-08,"-1,016,826.25","-1,237,129.79","-1,232,580.69","-4,549.10"
8,2026-01-09,"-796,216.83","29,987.35","21,556.09","8,431.26"
9,2026-01-10,"1,295,047.93","1,946,708.92","1,942,216.69","4,492.23"


### 3. Générateur automatique de rapport d'anomalies (Mismatches)

Ce module va simuler la détection d'anomalies entre différentes sources de données : `MTM Comptable`, *ici on crée artificiellement, mais on imagine que les données sont importées*) ou des ruptures de réconciliation.


In [194]:
df_accounting_mtm = df_daily_pnl[['Date', 'MTM_Total_EUR']].copy()
df_accounting_mtm = df_accounting_mtm.rename(columns={'MTM_Total_EUR': 'MTM_Comptable_EUR'})

def introduce_break(row):
    # Comparaison directe avec chaîne de caractères ISO
    if str(row['Date'])[:10] == '2026-01-03':
        return row['MTM_Comptable_EUR'] * 1.05  # +5% mismatch
    elif str(row['Date'])[:10] == '2026-01-05':
        return row['MTM_Comptable_EUR'] * 0.98  # -2% mismatch
    return row['MTM_Comptable_EUR']

df_accounting_mtm['MTM_Comptable_EUR'] = df_accounting_mtm.apply(introduce_break, axis=1)

# Fusionner les données pour la réconciliation
df_reconciliation = pd.merge(
    df_daily_pnl[['Date', 'MTM_Total_EUR', 'PnL_Total_MTM_EUR', 'PnL_Effet_Prix_EUR', 'PnL_FX_Carry_EUR']],
    df_accounting_mtm,
    on='Date',
    how='left'
)

df_reconciliation['Ec_Reconciliation_MTM_EUR'] = df_reconciliation['MTM_Total_EUR'] - df_reconciliation['MTM_Comptable_EUR']

# définition du seuil d'anomalie
anomaly_threshold = 10000 # 10,000 EUR

df_reconciliation['Anomalie'] = np.where(
    abs(df_reconciliation['Ec_Reconciliation_MTM_EUR']) > anomaly_threshold,
    'Oui', 'Non'
)

df_anomalies = df_reconciliation[df_reconciliation['Anomalie'] == 'Oui']

print("Rapport de Réconciliation Quotidienne MTM:")
display(df_reconciliation)

if not df_anomalies.empty:
    print(f"\nAnomalies détectées (seuil > {anomaly_threshold} EUR):")
    display(df_anomalies)
else:
    print("\nAucune anomalie détectée pour le seuil défini.")

Rapport de Réconciliation Quotidienne MTM:


,Date,MTM_Total_EUR,PnL_Total_MTM_EUR,PnL_Effet_Prix_EUR,PnL_FX_Carry_EUR,MTM_Comptable_EUR,Ec_Reconciliation_MTM_EUR,Anomalie
0,2026-01-01,"586,470.80","-494,109.43","-494,109.43",NaN,"586,470.80",0.00,Non
1,2026-01-02,"286,049.92","-27,409.44","-29,742.82","2,333.38","286,049.92",0.00,Non
2,2026-01-03,"34,286.05","-251,763.87","-253,546.92","1,783.05","36,000.35","-1,714.30",Non
3,2026-01-04,"-107,834.52","270,378.49","270,378.49",NaN,"-107,834.52",0.00,Non
4,2026-01-05,"533,401.29","-14,772.06","-18,618.42","3,846.37","522,733.26","10,668.03",Oui
5,2026-01-06,"120,252.75","-110,487.55","-85,404.12","-25,083.43","120,252.75",0.00,Non
6,2026-01-07,"645,341.68","348,057.68","358,447.11","-10,389.43","645,341.68",0.00,Non
7,2026-01-08,"-1,016,826.25","-1,237,129.79","-1,232,580.69","-4,549.10","-1,016,826.25",0.00,Non
8,2026-01-09,"-796,216.83","29,987.35","21,556.09","8,431.26","-796,216.83",0.00,Non
9,2026-01-10,"1,295,047.93","1,946,708.92","1,942,216.69","4,492.23","1,295,047.93",0.00,Non



Anomalies détectées (seuil > 10000 EUR):


,Date,MTM_Total_EUR,PnL_Total_MTM_EUR,PnL_Effet_Prix_EUR,PnL_FX_Carry_EUR,MTM_Comptable_EUR,Ec_Reconciliation_MTM_EUR,Anomalie
4,2026-01-05,"533,401.29","-14,772.06","-18,618.42","3,846.37","522,733.26","10,668.03",Oui


In [193]:
# RESET Variables
# Get a list of all user-defined variables
#user_vars = [var for var in globals() if not var.startswith('_') and var != 'In' and var != 'Out' and var != 'get_ipython']

# Delete each user-defined variable
#for var in user_vars:
#    del globals()[var]

#print("All user-defined variables have been cleared.")